In [1]:
import torch, time
from torchvision.models import resnet18
from torch.profiler import profile, ProfilerActivity, record_function

In [4]:
model = resnet18()
parameters = [a for a, b in model.named_parameters()]
random_input = torch.randn(size=(5,3,224, 224)) #3 is the channel count

In [11]:
activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities += [ProfilerActivity.CUDA]
    model = model.to("cuda")
    random_input = random_input.to("cuda")
with profile(activities=activities, profile_memory=True, record_shapes=True) as prof:
    with record_function("inference"):
        logits = model(random_input)

In [12]:
print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                aten::cudnn_convolution        46.79%        1.123s        55.98%        1.343s      67.151ms       1.375ms        78.24%       8.929ms     446.449us           0 B           0 B      49.78 MB      49.78 M

In [13]:
print(prof.key_averages().table(sort_by="self_cuda_time_total", row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              inference         0.00%       0.000us         0.00%       0.000us       0.000us     593.861ms     33801.11%     593.861ms     296.930ms           0 B           0 B           0 B           0 

In [16]:
print(prof.key_averages().table(sort_by="self_cuda_memory_total", row_limit=20))

AttributeError: 'FunctionEventAvg' object has no attribute 'self_device_memory_total'